# CNN Model

### Import packages and load data

In [1]:
%reload_ext autoreload
%autoreload 2
%aimport -numpy, -pandas, -matplotlib

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import matplotlib.pyplot as plt
# use same font as latex
# plt.rc('text', usetex=True)

plt.rc('font', family='serif')

import pandas as pd
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  
from dataset.utils.fungi_vis import FungiTasticVis
from dataset.fungi import FungiTastic
# import SimpleNamespace

#  fix random seeds for reproducibility
import random
import numpy as np

import tensorflow as tf
from tqdm import tqdm

from pathlib import Path
from collections import Counter

random.seed(0)
np.random.seed(0)

/Users/jeremycui/Documents/UCB_MIDS/DATASCI207/fungitastic-classification-datasci207-Fall-2025/.dataset_demo_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
fraction = 0.8

'''
valset = FungiTasticVis(
        root=f"{os.getcwd()}/dataset/FungiTastic/",
        split='val',
        size='300',
        task='open',
        data_subset='Mini',
        take_fraction=0.8,
        transform=None,
)

trainset = FungiTasticVis(
        root=f"{os.getcwd()}/dataset/FungiTastic/",
        split='train',
        size='300',
        task='closed',
        data_subset='Mini',
        take_fraction=0.8,
        transform=None,
)
'''
root=f"{os.getcwd()}/dataset/FungiTastic/"
trainset, testset = FungiTastic.from_path_train_test(
    root=root,
    data_subset='Mini',
    split='train',             # which CSV to read from (Train/Val/Test filenames)
    size='300',
    task='closed',
    take_fraction=fraction,       # take first 80% of rows from the CSV
    train_test_split=0.8,    # of that 80% -> 70% train, 30% test
    shuffle=True,            # optional: shuffle before splitting
    random_state=0,          # reproducible shuffle
    transform=None
)

valset = FungiTasticVis(
        root=f"{os.getcwd()}/dataset/FungiTastic/",
        split='val',
        size='300',
        task='open',
        data_subset='Mini',
        take_fraction=fraction,
        transform=None,
)

In [3]:
os.getcwd()

'/Users/jeremycui/Documents/UCB_MIDS/DATASCI207/fungitastic-classification-datasci207-Fall-2025'

In [4]:
BASE_METADATA_PATH = f"{os.getcwd()}/baselines/closed_set/FungiTastic/metadata/FungiTastic-Mini"

train_metadata = pd.read_csv(f"{BASE_METADATA_PATH}/FungiTastic-Mini-Train.csv")
val_metadata  = pd.read_csv(f"{BASE_METADATA_PATH}/FungiTastic-Mini-ClosedSet-Val.csv")

In [5]:
train_test_n = int(len(train_metadata) * fraction)
train_n = int(len(train_metadata) * fraction * 0.8)
val_n = int(len(val_metadata) * fraction)

In [6]:
train_metadata = train_metadata[['species','year','month','day','habitat','countryCode','hasCoordinate','iucnRedListCategory','substrate','latitude','longitude','coorUncert','region','district','metaSubstrate','poisonous','elevation','landcover','biogeographicalRegion']]
train_test_metadata = train_metadata.iloc[:train_test_n].reset_index(drop=True)

train_metadata = train_test_metadata.iloc[:train_n].reset_index(drop=True)
test_metadata = train_test_metadata.iloc[train_n:].reset_index(drop=True)

In [7]:
val_metadata = val_metadata[['species', 'year','month','day','habitat','countryCode','hasCoordinate','iucnRedListCategory','substrate','latitude','longitude','coorUncert','region','district','metaSubstrate','poisonous','elevation','landcover','biogeographicalRegion']]
val_metadata = val_metadata.iloc[:val_n].reset_index(drop=True)

In [8]:
print("train_metadata size: ", train_metadata.shape)
print("val_metadata size: ", val_metadata.shape)
print("test_metadata size: ", test_metadata.shape)

train_metadata size:  (29978, 19)
val_metadata size:  (7529, 19)
test_metadata size:  (7495, 19)


### Data preprocessing

In this code chunk, I am defining functions that first extract the training paths and labels from the datasets loaded above, and then define a function that creates a tf.dataset so that not all images need to be loaded into memory at once (will slow computer or crash).

In [9]:
def extract_paths_and_labels(ds):
    """Extract file paths and labels."""
    paths, labels = [], []
    for i in tqdm(range(len(ds)), desc="Indexing"):
        _, y, p = ds[i]  # (PIL_image, label, path)
        
        paths.append(str(p))
        if y:
            labels.append(int(y))  
    return paths, labels


def get_top_labels(labels, top_n=20):
    label_counts = Counter(labels)
    top_labels = [label for label, _ in label_counts.most_common(top_n)]
    print(f"Top {top_n} species (labels): {top_labels}")
    return set(top_labels)


def filter_by_labels(paths, labels, metadata, allowed_labels):
    # metadata must be a numpy array or df -> convert to numpy first
    metadata_np = np.asarray(metadata, dtype=np.float32)

    filtered = []
    for i, (p, y) in enumerate(zip(paths, labels)):
        if y in allowed_labels:
            if i >= len(metadata_np):
                continue
            else:
                m = metadata_np[i]      # row i, shape (20,)
            filtered.append((p, m, y))

    f_paths, m_data, f_labels = zip(*filtered)
    return list(f_paths), np.array(m_data, dtype=np.float32), list(f_labels)

In [10]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

train_meta_numeric = ohe.fit_transform(train_metadata)
val_meta_numeric = ohe.transform(val_metadata)
test_meta_numeric = ohe.transform(test_metadata)

In [11]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# (Optional but recommended) standardize
scaler = StandardScaler(with_mean=False)   # IMPORTANT because data is sparse

train_meta_scaled = scaler.fit_transform(train_meta_numeric)
val_meta_scaled   = scaler.transform(val_meta_numeric)
test_meta_scaled  = scaler.transform(test_meta_numeric)

In [15]:
# Apply PCA
pca = PCA(n_components=512, random_state=1234)

trainset = pca.fit_transform(train_meta_scaled)
valset   = pca.transform(val_meta_scaled)
testset  = pca.transform(test_meta_scaled)

: 

In [12]:
# Train
train_paths, train_labels = extract_paths_and_labels(trainset)
top_labels = get_top_labels(train_labels, top_n=10)
train_paths, train_dataset, train_labels = filter_by_labels(train_paths, train_labels, train_meta_numeric, top_labels)

Indexing: 100%|██████████| 29978/29978 [00:34<00:00, 861.75it/s]


Top 10 species (labels): [83, 53, 44, 33, 39, 29, 179, 109, 85, 177]


In [13]:
# Validation?
val_paths, val_labels = extract_paths_and_labels(valset)

Indexing: 100%|██████████| 7560/7560 [00:09<00:00, 771.61it/s]


In [14]:
val_paths, val_dataset, val_labels = filter_by_labels(val_paths, val_labels, val_meta_numeric, top_labels)

In [15]:
# Validation
test_paths, test_labels = extract_paths_and_labels(testset)

Indexing: 100%|██████████| 7495/7495 [00:08<00:00, 859.84it/s]


In [16]:
test_paths, test_dataset, test_labels = filter_by_labels(test_paths, test_labels, test_meta_numeric, top_labels)

In [17]:
autotune   = tf.data.AUTOTUNE
batch_size      = 64
h, w = 224, 224

def decode_and_resize_from_path(path):
    bytes_ = tf.io.read_file(path)
    img = tf.image.decode_image(bytes_, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, [h, w],
                          method=tf.image.ResizeMethod.LANCZOS5)
    img = tf.clip_by_value(img, 0.0, 255.0)
    img = tf.cast(img, tf.float32) / 255.0
    return img

In [18]:
top_labels = np.unique(train_labels)

label_map = {old: new for new, old in enumerate(top_labels)}

label_map_tf = tf.lookup.StaticHashTable(
    initializer=tf.lookup.KeyValueTensorInitializer(
        keys=tf.constant(list(label_map.keys()),   dtype=tf.int64),
        values=tf.constant(list(label_map.values()), dtype=tf.int64),
    ),
    default_value=tf.constant(-1, dtype=tf.int64)  
)

def make_supervised_ds(paths, metadata, labels, training=True, shuffle_buf=10_000):
    """
    Creates a tf.data.Dataset:
      - reads (path, label)
      - decodes/resizes image
      - (re)maps label via label_map_tf to contiguous [0..n_classes-1]
      - optional flip augmentation
      - batches & prefetches
    """
    ds = tf.data.Dataset.from_tensor_slices((paths, metadata, labels))
    if training:
        ds = ds.shuffle(shuffle_buf, reshuffle_each_iteration=True)

    def _load_and_map(p, m, y):
        x = decode_and_resize_from_path(p)                      
        y = tf.cast(y, tf.int64)
        y = label_map_tf.lookup(y)                              

        tf.debugging.assert_greater_equal(y, tf.constant(0, tf.int64),
                                          message="Label not in label_map (got -1).")
        return x, m, tf.cast(y, tf.int32)

    autotune = tf.data.AUTOTUNE
    ds = ds.map(_load_and_map, num_parallel_calls=autotune)
    # ds = ds.map(_load_and_map_v2, num_parallel_calls=autotune)

    def _normalize(x, m, y):
        x = tf.image.convert_image_dtype(x, tf.float32)  # scales to [0,1]
        return x, m, y

    ds = ds.map(_normalize, num_parallel_calls=autotune)
    ds = ds.ignore_errors()
    
    def maybe_augment_image(x, m, y, augment_prob=0.1):
        # Draw a random number in [0,1)
        rand_val = tf.random.uniform([], 0, 1)
        
        def augment_fn():
            # Apply brightness or other augmentations here
            image_aug = tf.image.random_brightness(x, max_delta=0.2)
            return tf.clip_by_value(image_aug, 0.0, 1.0)
        
        # Apply augmentation with given probability
        return tf.cond(rand_val < augment_prob, augment_fn, lambda: x), m, y
    
    if training:
        ds = ds.map(lambda x, m, y: (tf.image.random_flip_left_right(x), m, y),
                    num_parallel_calls=autotune)
        ds = ds.map(maybe_augment_image, num_parallel_calls=autotune)
    
    def _pack_inputs(x, m, y):
        return {"image": x, "metadata": m}, y

    ds = ds.map(_pack_inputs)

    batch_size = 64
    ds = ds.batch(batch_size, drop_remainder=training).prefetch(autotune)

    opts = tf.data.Options(); opts.experimental_deterministic = False
    return ds.with_options(opts)

In [19]:
train_ds = make_supervised_ds(train_paths, train_dataset, train_labels, training=True)

In [20]:
val_ds   = make_supervised_ds(val_paths, val_dataset, val_labels, training=False)

In [21]:
test_ds   = make_supervised_ds(test_paths, test_dataset, test_labels, training=False)

In [23]:
#Sanity check
for images_metadata, labels in train_ds.take(1):
    print("Image batch shape:", images_metadata['image'].shape)
    print("Metadata batch shape:", images_metadata['metadata'].shape)
    print("Label batch shape:", labels.shape)
    print("dtype:", images_metadata['image'].dtype)
    print("Min pixel value:", tf.reduce_min(images_metadata['image']).numpy())
    print("Max pixel value:", tf.reduce_max(images_metadata['image']).numpy())

Image batch shape: (64, 224, 224, 3)
Metadata batch shape: (64, 33695)
Label batch shape: (64,)
dtype: <dtype: 'float32'>
Min pixel value: 0.0
Max pixel value: 1.0


### Baseline Model: Majority Class Predictor

In [21]:
majority_label = label_map[Counter(train_labels).most_common(1)[0][0]]
print("Majority class:", majority_label)

Majority class: 13


In [22]:
train_ds

<_OptionsDataset element_spec=({'image': TensorSpec(shape=(64, 224, 224, 3), dtype=tf.float32, name=None), 'metadata': TensorSpec(shape=(64, 48132), dtype=tf.float32, name=None)}, TensorSpec(shape=(64,), dtype=tf.int32, name=None))>

In [25]:
def majority_accuracy(dataset, majority_label):
    total = 0
    correct = 0
    for i, (i_m, y) in enumerate(dataset):
        y_np = y.numpy()
        pred = np.full_like(y_np, fill_value=majority_label)
        correct += (pred == y_np).sum()
        total += y_np.size
    return correct, total

maj_train_acc = majority_accuracy(train_ds, majority_label)
# maj_val_acc   = majority_accuracy(val_ds, majority_label)
# print(f"Majority baseline — train = {maj_train_acc:.3f}, val={maj_val_acc:.3f}")

In [26]:
maj_train_acc

(np.int64(1639), 17856)

### CNN baseline + metadata

In [27]:
n_classes = len(np.unique(train_labels))

# define early stopping class
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='accuracy', 
    verbose=1,
    patience=5,
    mode='max',
    restore_best_weights=True
)

In [ ]:
from keras import layers, models, Input

tf.random.set_seed(1234)
np.random.seed(1234)

image_input = Input(shape=(224, 224, 3))


#-----------------------------------------------
x = layers.Conv2D(
    filters=32, kernel_size=(3, 3), strides=(1,1), padding='same', activation='relu', name='conv_1'
)(image_input)
x = layers.MaxPooling2D(pool_size=(2,2))(x)
x = layers.Conv2D(64, (3, 3), activation='relu', name='conv_2')(x)
x = layers.MaxPooling2D()(x)
x = layers.Conv2D(128, (3, 3), activation='relu', name='conv_3')(x)
x = layers.MaxPooling2D(pool_size=(2,2))(x)
x = layers.Dropout(rate=0.4)(x)
x = layers.GlobalAveragePooling2D()(x)
#-------------------------------------------------

# Metadata branch --------------------------------
metadata_input = Input(shape=(33695,))

m = layers.Dense(256, activation='relu')(metadata_input)
m = layers.Dropout(0.2)(m)

m = layers.Dense(128, activation='relu')(m)
m = layers.Dropout(0.2)(m)

m = layers.Dense(64, activation='relu')(m)   # bottleneck
#--------------------------------------------------

# Combine branches --------------------------------
combined = layers.concatenate([x, m])
z = layers.Dense(128, activation='relu')(combined)
z = layers.Dense(64, activation='relu')(z)

# Final output softmax layer
output = layers.Dense(10, activation='softmax')(z)
#--------------------------------------------------

# Build model
model_relu = models.Model(
    inputs={"image": image_input, "metadata": metadata_input}, outputs=output
)

model_relu.summary()

model_relu.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=3, restore_best_weights=True
)

/Users/jeremycui/Documents/UCB_MIDS/DATASCI207/fungitastic-classification-datasci207-Fall-2025/.dataset_demo_venv/lib/python3.12/site-packages/keras/src/models/functional.py:107: UserWarning: When providing `inputs` as a dict, all keys in the dict must match the names of the corresponding tensors. Received key 'image' mapping to value <KerasTensor shape=(None, 224, 224, 3), dtype=float32, sparse=False, ragged=False, name=keras_tensor_17> which has name 'keras_tensor_17'. Change the tensor name to 'image' (via `Input(..., name='image')`)
  warnings.warn(
/Users/jeremycui/Documents/UCB_MIDS/DATASCI207/fungitastic-classification-datasci207-Fall-2025/.dataset_demo_venv/lib/python3.12/site-packages/keras/src/models/functional.py:107: UserWarning: When providing `inputs` as a dict, all keys in the dict must match the names of the corresponding tensors. Received key 'metadata' mapping to value <KerasTensor shape=(None, 33695), dtype=float32, sparse=False, ragged=False, name=keras_tensor_26> w

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_1 (Conv2D)     │ (None, 224, 224,  │      1,344 │ input_layer_2[0]… │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 112, 112,  │          0 │ conv_1[0][0]      │
│ (MaxPooling2D)      │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_2 (Conv2D)     │ (None, 110, 110,  │     27,712 │ max_pooling2d_3[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 55, 55,    │          0 │ conv_2[0][0]      │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_3 (Conv2D)     │ (None, 53, 53,    │     73,856 │ max_pooling2d_4[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, 33695)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_5     │ (None, 26, 26,    │          0 │ conv_3[0][0]      │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 256)       │  8,626,176 │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 26, 26,    │          0 │ max_pooling2d_5[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 256)       │          0 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ dropout_2[0][0]   │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 64)        │     16,448 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 192)       │          0 │ global_average_p… │
│ (Concatenate)       │                   │            │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 128)       │     24,704 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 64)        │      8,256 │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 10)        │        650 │ dense_8[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 8,779,146 (33.49 MB)

 Trainable params: 8,779,146 (33.49 MB)

 Non-trainable params: 0 (0.00 B)

In [27]:
history_relu = model_relu.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stopping],  
    verbose=1
)

Epoch 1/10
    120/Unknown 272s 2s/step - accuracy: 0.1389 - loss: 2.2807

/Users/jeremycui/Documents/UCB_MIDS/DATASCI207/fungitastic-classification-datasci207-Fall-2025/.dataset_demo_venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


120/120 ━━━━━━━━━━━━━━━━━━━━ 298s 2s/step - accuracy: 0.1419 - loss: 2.2571 - val_accuracy: 0.0870 - val_loss: 2.2644
Epoch 2/10
120/120 ━━━━━━━━━━━━━━━━━━━━ 376s 3s/step - accuracy: 0.1603 - loss: 2.2254 - val_accuracy: 0.1202 - val_loss: 2.2910
Epoch 3/10
120/120 ━━━━━━━━━━━━━━━━━━━━ 605s 5s/step - accuracy: 0.2567 - loss: 2.0296 - val_accuracy: 0.1230 - val_loss: 2.3752
Epoch 4/10
120/120 ━━━━━━━━━━━━━━━━━━━━ 365s 3s/step - accuracy: 0.4549 - loss: 1.5792 - val_accuracy: 0.1349 - val_loss: 2.5384
Epoch 5/10
120/120 ━━━━━━━━━━━━━━━━━━━━ 373s 3s/step - accuracy: 0.6395 - loss: 1.1422 - val_accuracy: 0.1102 - val_loss: 2.7821
Epoch 6/10
120/120 ━━━━━━━━━━━━━━━━━━━━ 998s 8s/step - accuracy: 0.7242 - loss: 0.8770 - val_accuracy: 0.0982 - val_loss: 2.8802
Epoch 7/10
120/120 ━━━━━━━━━━━━━━━━━━━━ 553s 5s/step - accuracy: 0.7663 - loss: 0.7267 - val_accuracy: 0.1014 - val_loss: 3.0060


In [28]:
test_loss, test_acc = model_relu.evaluate(test_ds)

30/30 ━━━━━━━━━━━━━━━━━━━━ 11s 350ms/step - accuracy: 0.1272 - loss: 2.5991


In [29]:
history_relu.history['accuracy']

[0.14191631972789764,
 0.16025309264659882,
 0.2567148804664612,
 0.45493283867836,
 0.6394628286361694,
 0.7241735458374023,
 0.766270637512207]

In [30]:
history_relu.history['val_accuracy']

[0.087025947868824,
 0.12015967816114426,
 0.12295409291982651,
 0.13493013381958008,
 0.11017964035272598,
 0.09820359200239182,
 0.10139720886945724]

In [31]:
test_acc

0.12717677652835846

In [24]:
from keras import layers, models, Input

tf.random.set_seed(1234)
np.random.seed(1234)

image_input = Input(shape=(224, 224, 3))


#-----------------------------------------------
x = layers.Conv2D(
    filters=48, kernel_size=(3, 3), strides=(1,1), padding='same', activation='gelu', name='conv_1'
)(image_input)
x = layers.MaxPooling2D(pool_size=(2,2))(x)
x = layers.Conv2D(64, (3, 3), activation='gelu', name='conv_2')(x)
x = layers.MaxPooling2D()(x)
x = layers.Conv2D(128, (3, 3), activation='gelu', name='conv_3')(x)
x = layers.MaxPooling2D(pool_size=(2,2))(x)
x = layers.Dropout(rate=0.4)(x)
x = layers.GlobalAveragePooling2D()(x)
#-------------------------------------------------

# Metadata branch --------------------------------
metadata_input = Input(shape=(31869,))

m = layers.Dense(256, activation='gelu')(metadata_input)
m = layers.Dropout(0.2)(m)

m = layers.Dense(128, activation='gelu')(m)
m = layers.Dropout(0.2)(m)

m = layers.Dense(64, activation='gelu')(m)   # bottleneck
#--------------------------------------------------

# Combine branches --------------------------------
combined = layers.concatenate([x, m])
z = layers.Dense(128, activation='gelu')(combined)
z = layers.Dense(64, activation='gelu')(z)

# Final output softmax layer
output = layers.Dense(10, activation='softmax')(z)
#--------------------------------------------------

# Build model
model_gelu = models.Model(
    inputs={"image": image_input, "metadata": metadata_input}, outputs=output
)

model_gelu.summary()

model_gelu.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=3, restore_best_weights=True
)

/Users/jeremycui/Documents/UCB_MIDS/DATASCI207/fungitastic-classification-datasci207-Fall-2025/.dataset_demo_venv/lib/python3.12/site-packages/keras/src/models/functional.py:107: UserWarning: When providing `inputs` as a dict, all keys in the dict must match the names of the corresponding tensors. Received key 'image' mapping to value <KerasTensor shape=(None, 224, 224, 3), dtype=float32, sparse=False, ragged=False, name=keras_tensor_19> which has name 'keras_tensor_19'. Change the tensor name to 'image' (via `Input(..., name='image')`)
  warnings.warn(
/Users/jeremycui/Documents/UCB_MIDS/DATASCI207/fungitastic-classification-datasci207-Fall-2025/.dataset_demo_venv/lib/python3.12/site-packages/keras/src/models/functional.py:107: UserWarning: When providing `inputs` as a dict, all keys in the dict must match the names of the corresponding tensors. Received key 'metadata' mapping to value <KerasTensor shape=(None, 31869), dtype=float32, sparse=False, ragged=False, name=keras_tensor_28> w

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_1 (Conv2D)     │ (None, 224, 224,  │      1,344 │ input_layer_2[0]… │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 112, 112,  │          0 │ conv_1[0][0]      │
│ (MaxPooling2D)      │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_2 (Conv2D)     │ (None, 110, 110,  │     27,712 │ max_pooling2d_3[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, 31869)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 55, 55,    │          0 │ conv_2[0][0]      │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 256)       │  8,158,720 │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_3 (Conv2D)     │ (None, 53, 53,    │     73,856 │ max_pooling2d_4[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 256)       │          0 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_5     │ (None, 26, 26,    │          0 │ conv_3[0][0]      │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 128)       │     32,896 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 26, 26,    │          0 │ max_pooling2d_5[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 128)       │          0 │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ dropout_3[0][0]   │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 64)        │      8,256 │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 192)       │          0 │ global_average_p… │
│ (Concatenate)       │                   │            │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 128)       │     24,704 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 64)        │      8,256 │ dense_9[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 10)        │        650 │ dense_10[0][0]    │
└─────────────────────┴───────────────────┴────────────┴─────────────────

 Total params: 8,336,394 (31.80 MB)

 Trainable params: 8,336,394 (31.80 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history_gelu = model_gelu.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/10
    114/Unknown 393s 3s/step - accuracy: 0.1279 - loss: 2.2805

/Users/jeremycui/Documents/UCB_MIDS/DATASCI207/fungitastic-classification-datasci207-Fall-2025/.dataset_demo_venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


114/114 ━━━━━━━━━━━━━━━━━━━━ 417s 4s/step - accuracy: 0.1332 - loss: 2.2571 - val_accuracy: 0.0902 - val_loss: 2.2572
Epoch 2/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 339s 3s/step - accuracy: 0.1640 - loss: 2.2062 - val_accuracy: 0.1547 - val_loss: 2.3038
Epoch 3/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 352s 3s/step - accuracy: 0.3450 - loss: 1.8060 - val_accuracy: 0.1138 - val_loss: 2.5470
Epoch 4/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 1877s 17s/step - accuracy: 0.6413 - loss: 1.1315 - val_accuracy: 0.1070 - val_loss: 2.9419
Epoch 5/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 339s 3s/step - accuracy: 0.7598 - loss: 0.7337 - val_accuracy: 0.1167 - val_loss: 2.9149


In [26]:
test_loss_gelu, test_acc_gelu = model_gelu.evaluate(test_ds)

29/29 ━━━━━━━━━━━━━━━━━━━━ 20s 667ms/step - accuracy: 0.1147 - loss: 2.3137


/Users/jeremycui/Documents/UCB_MIDS/DATASCI207/fungitastic-classification-datasci207-Fall-2025/.dataset_demo_venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


In [27]:
history_gelu.history['accuracy']

[0.13315217196941376,
 0.16399456560611725,
 0.3449728190898895,
 0.6413043737411499,
 0.759782612323761]

In [28]:
history_gelu.history['val_accuracy']

[0.09018120169639587,
 0.1546565592288971,
 0.11378002166748047,
 0.1070375069975853,
 0.11672987788915634]

In [29]:
test_acc_gelu

0.11474501341581345

In [34]:
from keras import layers, models, Input

tf.random.set_seed(1234)
np.random.seed(1234)

image_input = Input(shape=(224, 224, 3))


#-----------------------------------------------
x = layers.Conv2D(
    filters=32, kernel_size=(3, 3), strides=(1,1), padding='same', activation='gelu', name='conv_1'
)(image_input)
x = layers.MaxPooling2D(pool_size=(2,2))(x)
x = layers.Dropout(rate=0.3)(x)

x = layers.Conv2D(64, (3, 3), activation='gelu', name='conv_2')(x)
x = layers.MaxPooling2D()(x)
x = layers.Dropout(rate=0.3)(x)

x = layers.Conv2D(128, (3, 3), activation='gelu', name='conv_3')(x)
x = layers.MaxPooling2D(pool_size=(2,2))(x)
x = layers.Dropout(rate=0.4)(x)
x = layers.GlobalAveragePooling2D()(x)
#-------------------------------------------------

# Metadata branch --------------------------------
metadata_input = Input(shape=(31869,))

m = layers.Dense(512, activation='gelu')(metadata_input)
m = layers.Dropout(0.2)(m)

m = layers.Dense(256, activation='gelu')(m)
m = layers.Dropout(0.2)(m)

m = layers.Dense(128, activation='gelu')(m)
m = layers.Dropout(0.3)(m)
#--------------------------------------------------

# Combine branches --------------------------------
combined = layers.concatenate([x, m])

z = layers.Dense(256, activation='gelu')(combined)
z = layers.Dropout(0.6)(z)

z = layers.Dense(128, activation='gelu')(z)
z = layers.Dropout(0.6)(z)

z = layers.Dense(64, activation='gelu')(z)
z = layers.Dropout(0.5)(z)

# Final output softmax layer
output = layers.Dense(10, activation='softmax')(z)
#--------------------------------------------------

# Build model
model_gelu_plus_1m_plus_1z_more_dropouts = models.Model(
    inputs={"image": image_input, "metadata": metadata_input}, outputs=output
)

model_gelu_plus_1m_plus_1z_more_dropouts.summary()

model_gelu_plus_1m_plus_1z_more_dropouts.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy'])

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=3, restore_best_weights=True
)

/Users/jeremycui/Documents/UCB_MIDS/DATASCI207/fungitastic-classification-datasci207-Fall-2025/.dataset_demo_venv/lib/python3.12/site-packages/keras/src/models/functional.py:107: UserWarning: When providing `inputs` as a dict, all keys in the dict must match the names of the corresponding tensors. Received key 'image' mapping to value <KerasTensor shape=(None, 224, 224, 3), dtype=float32, sparse=False, ragged=False, name=keras_tensor_116> which has name 'keras_tensor_116'. Change the tensor name to 'image' (via `Input(..., name='image')`)
  warnings.warn(
/Users/jeremycui/Documents/UCB_MIDS/DATASCI207/fungitastic-classification-datasci207-Fall-2025/.dataset_demo_venv/lib/python3.12/site-packages/keras/src/models/functional.py:107: UserWarning: When providing `inputs` as a dict, all keys in the dict must match the names of the corresponding tensors. Received key 'metadata' mapping to value <KerasTensor shape=(None, 31869), dtype=float32, sparse=False, ragged=False, name=keras_tensor_127

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_10      │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_1 (Conv2D)     │ (None, 224, 224,  │        896 │ input_layer_10[0… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_15    │ (None, 112, 112,  │          0 │ conv_1[0][0]      │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_33          │ (None, 112, 112,  │          0 │ max_pooling2d_15… │
│ (Dropout)           │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_2 (Conv2D)     │ (None, 110, 110,  │     18,496 │ dropout_33[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_11      │ (None, 31869)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_16    │ (None, 55, 55,    │          0 │ conv_2[0][0]      │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_33 (Dense)    │ (None, 512)       │ 16,317,440 │ input_layer_11[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_34          │ (None, 55, 55,    │          0 │ max_pooling2d_16… │
│ (Dropout)           │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_36          │ (None, 512)       │          0 │ dense_33[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_3 (Conv2D)     │ (None, 53, 53,    │     73,856 │ dropout_34[0][0]  │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_34 (Dense)    │ (None, 256)       │    131,328 │ dropout_36[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_17    │ (None, 26, 26,    │          0 │ conv_3[0][0]      │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_37          │ (None, 256)       │          0 │ dense_34[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_35          │ (None, 26, 26,    │          0 │ max_pooling2d_17… │
│ (Dropout)           │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_35 (Dense)    │ (None, 128)       │     32,896 │ dropout_37[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ dropout_35[0][0]  │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_38          │ (None, 128)       │          0 │ dense_35[0][0]  

 Total params: 16,682,506 (63.64 MB)

 Trainable params: 16,682,506 (63.64 MB)

 Non-trainable params: 0 (0.00 B)

In [35]:
history_gelu_plus_1m_plus_1z_more_dropouts = model_gelu_plus_1m_plus_1z_more_dropouts.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/10
    114/Unknown 316s 3s/step - accuracy: 0.1278 - loss: 2.2911

/Users/jeremycui/Documents/UCB_MIDS/DATASCI207/fungitastic-classification-datasci207-Fall-2025/.dataset_demo_venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


114/114 ━━━━━━━━━━━━━━━━━━━━ 334s 3s/step - accuracy: 0.1268 - loss: 2.2662 - val_accuracy: 0.0902 - val_loss: 2.2712
Epoch 2/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 314s 3s/step - accuracy: 0.1371 - loss: 2.2491 - val_accuracy: 0.0889 - val_loss: 2.2909
Epoch 3/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 430s 4s/step - accuracy: 0.1818 - loss: 2.1568 - val_accuracy: 0.0940 - val_loss: 2.3222
Epoch 4/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 290s 3s/step - accuracy: 0.2683 - loss: 1.9461 - val_accuracy: 0.0805 - val_loss: 2.4680
Epoch 5/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 316s 3s/step - accuracy: 0.4211 - loss: 1.6476 - val_accuracy: 0.0809 - val_loss: 2.6473
Epoch 6/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 317s 3s/step - accuracy: 0.5628 - loss: 1.3475 - val_accuracy: 0.0868 - val_loss: 2.7022


In [36]:
test_loss_gelu_plus_1m_plus_1z_more_dropouts, test_acc_gelu_plus_1m_plus_1z_more_dropouts = model_gelu_plus_1m_plus_1z_more_dropouts.evaluate(test_ds)

29/29 ━━━━━━━━━━━━━━━━━━━━ 18s 613ms/step - accuracy: 0.1197 - loss: 2.3022


/Users/jeremycui/Documents/UCB_MIDS/DATASCI207/fungitastic-classification-datasci207-Fall-2025/.dataset_demo_venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


In [37]:
history_gelu_plus_1m_plus_1z_more_dropouts.history['accuracy']

[0.12676630914211273,
 0.13709239661693573,
 0.1817934811115265,
 0.26834240555763245,
 0.421059787273407,
 0.562771737575531]

In [38]:
history_gelu_plus_1m_plus_1z_more_dropouts.history['val_accuracy']

[0.09018120169639587,
 0.08891697973012924,
 0.09397387504577637,
 0.08048883080482483,
 0.08091024309396744,
 0.08680994808673859]

In [39]:
test_acc_gelu_plus_1m_plus_1z_more_dropouts

0.11973392218351364

In [40]:
from sklearn.metrics import f1_score

# Predict
y_prob = model_gelu_plus_1m_plus_1z_more_dropouts.predict(test_ds)

# Convert predictions
y_pred = np.argmax(y_prob, axis=1)

# Extract true labels
y_true = np.concatenate([y for _, y in test_ds])

# If one-hot encoded
if y_true.ndim > 1:
    y_true = np.argmax(y_true, axis=1)

# Compute F1
f1 = f1_score(y_true, y_pred, average='macro')
print("Test F1:", f1)

29/29 ━━━━━━━━━━━━━━━━━━━━ 13s 446ms/step
Test F1: 0.07672039544600044


In [ ]:
from keras import layers, models, Input

tf.random.set_seed(1234)
np.random.seed(1234)

image_input = Input(shape=(224, 224, 3))

#-----------------------------------------------
x = layers.Conv2D(
    filters=48, kernel_size=(3, 3), strides=(1,1), padding='same', activation='gelu', name='conv_1'
)(image_input)
x = layers.MaxPooling2D(pool_size=(2,2))(x)
x = layers.Dropout(rate=0.1)(x)

x = layers.Conv2D(64, (3, 3), strides=(1,1), padding='same', activation='gelu', name='conv_2')(x)
x = layers.MaxPooling2D(pool_size=(2,2))(x)
x = layers.Dropout(rate=0.1)(x)

x = layers.Conv2D(128, (3, 3), strides=(1,1), padding='same', activation='gelu', name='conv_3')(x)
x = layers.MaxPooling2D(pool_size=(2,2))(x)
x = layers.Dropout(rate=0.2)(x)

x = layers.Conv2D(256, (3, 3), strides=(1,1), padding='same', activation='gelu', name='conv_4')(x)
x = layers.MaxPooling2D(pool_size=(2,2))(x)
x = layers.Dropout(rate=0.2)(x)

x = layers.GlobalAveragePooling2D()(x)
#-------------------------------------------------

# Metadata branch --------------------------------
metadata_input = Input(shape=(31869,))

# m = layers.Dense(5096, activation='gelu')(metadata_input)
# m = layers.BatchNormalization()(m)
# m = layers.Dropout(0.3)(m)

'''
m = layers.Dense(2048, activation='gelu')(metadata_input)
m = layers.BatchNormalization()(m)
m = layers.Dropout(0.2)(m)
'''

m = layers.Dense(1024, activation='gelu')(metadata_input)
m = layers.BatchNormalization()(m)
m = layers.Dropout(0.2)(m)

m = layers.Dense(512, activation='gelu')(m)
m = layers.Dropout(0.2)(m)

m = layers.Dense(256, activation='gelu')(m)
m = layers.Dropout(0.2)(m)

m = layers.Dense(128, activation='gelu')(m)
m = layers.Dropout(0.3)(m)

m = layers.Dense(64, activation='gelu')(m)   # bottleneck
#--------------------------------------------------

# Combine branches --------------------------------
combined = layers.concatenate([x, m])

z = layers.Dense(512, activation='gelu')(combined)
z = layers.Dropout(0.4)(z)

z = layers.Dense(128, activation='gelu')(z)
z = layers.Dropout(0.3)(z)

z = layers.Dense(64, activation='gelu')(z)
z = layers.Dropout(0.3)(z)

# Final output softmax layer
output = layers.Dense(10, activation='softmax')(z)
#--------------------------------------------------

# Build model
model_gelu_enhanced_v2 = models.Model(
    inputs={"image": image_input, "metadata": metadata_input}, outputs=output
)

model_gelu_enhanced_v2.summary()

model_gelu_enhanced_v2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=3, restore_best_weights=True
)

/Users/jeremycui/Documents/UCB_MIDS/DATASCI207/fungitastic-classification-datasci207-Fall-2025/.dataset_demo_venv/lib/python3.12/site-packages/keras/src/models/functional.py:107: UserWarning: When providing `inputs` as a dict, all keys in the dict must match the names of the corresponding tensors. Received key 'image' mapping to value <KerasTensor shape=(None, 224, 224, 3), dtype=float32, sparse=False, ragged=False, name=keras_tensor_172> which has name 'keras_tensor_172'. Change the tensor name to 'image' (via `Input(..., name='image')`)
  warnings.warn(
/Users/jeremycui/Documents/UCB_MIDS/DATASCI207/fungitastic-classification-datasci207-Fall-2025/.dataset_demo_venv/lib/python3.12/site-packages/keras/src/models/functional.py:107: UserWarning: When providing `inputs` as a dict, all keys in the dict must match the names of the corresponding tensors. Received key 'metadata' mapping to value <KerasTensor shape=(None, 31869), dtype=float32, sparse=False, ragged=False, name=keras_tensor_186

Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_14      │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_1 (Conv2D)     │ (None, 224, 224,  │      1,344 │ input_layer_14[0… │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_22    │ (None, 112, 112,  │          0 │ conv_1[0][0]      │
│ (MaxPooling2D)      │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_52          │ (None, 112, 112,  │          0 │ max_pooling2d_22… │
│ (Dropout)           │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_2 (Conv2D)     │ (None, 112, 112,  │     27,712 │ dropout_52[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_23    │ (None, 56, 56,    │          0 │ conv_2[0][0]      │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_53          │ (None, 56, 56,    │          0 │ max_pooling2d_23… │
│ (Dropout)           │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_15      │ (None, 31869)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_3 (Conv2D)     │ (None, 56, 56,    │     73,856 │ dropout_53[0][0]  │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_48 (Dense)    │ (None, 1024)      │ 32,634,880 │ input_layer_15[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_24    │ (None, 28, 28,    │          0 │ conv_3[0][0]      │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_56          │ (None, 1024)      │          0 │ dense_48[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_54          │ (None, 28, 28,    │          0 │ max_pooling2d_24… │
│ (Dropout)           │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_49 (Dense)    │ (None, 512)       │    524,800 │ dropout_56[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_4 (Conv2D)     │ (None, 28, 28,    │    295,168 │ dropout_54[0][0]  │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_57          │ (None, 512)       │          0 │ dense_49[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_25    │ (None, 14, 14,    │          0 │ conv_4[0][0]      │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 33,944,458 (129.49 MB)

 Trainable params: 33,944,458 (129.49 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history_gelu_enhanced_v2 = model_gelu_enhanced_v2.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/10
    114/Unknown 546s 5s/step - accuracy: 0.1302 - loss: 2.2851

/Users/jeremycui/Documents/UCB_MIDS/DATASCI207/fungitastic-classification-datasci207-Fall-2025/.dataset_demo_venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


114/114 ━━━━━━━━━━━━━━━━━━━━ 572s 5s/step - accuracy: 0.1318 - loss: 2.2601 - val_accuracy: 0.0902 - val_loss: 2.2674
Epoch 2/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 775s 7s/step - accuracy: 0.1485 - loss: 2.2389 - val_accuracy: 0.1028 - val_loss: 2.2824
Epoch 3/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 553s 5s/step - accuracy: 0.2212 - loss: 2.0572 - val_accuracy: 0.0961 - val_loss: 2.4645
Epoch 4/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 612s 5s/step - accuracy: 0.3868 - loss: 1.7079 - val_accuracy: 0.1256 - val_loss: 2.6649
Epoch 5/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 588s 5s/step - accuracy: 0.5671 - loss: 1.3277 - val_accuracy: 0.1138 - val_loss: 2.7887
Epoch 6/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 14331s 127s/step - accuracy: 0.7061 - loss: 1.0170 - val_accuracy: 0.0809 - val_loss: 2.8845
Epoch 7/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 32221s 284s/step - accuracy: 0.7637 - loss: 0.7971 - val_accuracy: 0.1066 - val_loss: 2.9091


: 

In [26]:
test_loss_gelu_enhanced, test_acc_gelu_enhanced = model_gelu_enhanced.evaluate(test_ds)

18/18 ━━━━━━━━━━━━━━━━━━━━ 22s 1s/step - accuracy: 0.2060 - loss: 1.6016


In [27]:
history_gelu_enhanced.history['accuracy']

[0.2095070481300354,
 0.22117076814174652,
 0.22887323796749115,
 0.24845950305461884]

In [28]:
history_gelu_enhanced.history['val_accuracy']

[0.26745718717575073,
 0.14163373410701752,
 0.25691699981689453,
 0.2667984068393707]

In [29]:
test_acc_gelu_enhanced

0.20598910748958588